# **Cadillac Tyre Degradation Analysis - Japan Race 2026**

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import fastf1

#### Loading Japan Race 2026 data:

In [ ]:
session = fastf1.get_session(2026, 'Japan', 'R')
session.event

In [ ]:
session.load()
session.results

#### Retrieving lap data of Bottas, Perez and Competitors (Hulkendberg, Bortoleto):

In [ ]:
drivers = session.laps.pick_drivers(['BOT', 'PER', 'HUL', 'BOR'])

## **Visualising all drivers stints**

In [ ]:
driver_stints = (drivers
                 .groupby(['Driver', 'Stint', 'Compound'])
                 .size()
                 .reset_index(name='LapCount'))

stints_fig, stints_ax = plt.subplots()

compound_colours = {
    'HARD' : 'white',
    'MEDIUM' : 'yellow',
    'SOFT' : 'red',
    'INTERMEDIATE' : 'green',
    'WET' : 'blue'
}

legend_elements=[
    Patch(facecolor='White', edgecolor='black', label="HARD"),
    Patch(facecolor='Yellow', edgecolor='black', label="MEDIUM"),
    Patch(facecolor='Red', edgecolor='black', label="SOFT"),
    Patch(facecolor='Green', edgecolor='black', label="INTERMEDIATE"),
    Patch(facecolor='Blue', edgecolor='black', label="WET")
]

for driver, group in driver_stints.groupby('Driver'):
    left = 0

    for _, row in group.iterrows():
        stints_ax.barh(
            driver,
            row['LapCount'],
            left=left,
            color=compound_colours.get(row['Compound']),
            edgecolor='black',
        )
        left += row['LapCount']
    
    

plt.xlabel('Laps')
plt.ylabel('Drivers')
plt.title('Driver Stints')
plt.legend(handles=legend_elements, title='Compounds', bbox_to_anchor=(1.05, 1), loc='best')
plt.show()

## **Laptimes of Selected Drivers When Using Hard Compounds Compared to Grid Average**

#### Calculating Lap Time Averages of All Drivers and Selected Drivers on Hard Compounds

In [ ]:
# Calculating the average grid-wide lap time of all laps with hard compound excluding in-laps, out-laps, yellow flags, safety cars and the first lap (0 days 00:01:41.094228813 average when you include all laps with hard compound)
all_drivers_on_hard = session.laps[
    (session.laps['Compound'] == 'HARD') & 
    (session.laps['PitInTime'].isna()) & 
    (session.laps['PitOutTime'].isna()) & 
    (session.laps['TrackStatus'] == '1') &
    (session.laps['LapNumber'] > 1)
]
lap_average_on_hard = all_drivers_on_hard['LapTime'].mean()

selected_drivers_on_hard = drivers[
    (drivers['Compound'] == 'HARD') &
    (session.laps['PitInTime'].isna()) & 
    (session.laps['PitOutTime'].isna()) & 
    (session.laps['TrackStatus'] == '1') &
    (session.laps['LapNumber'] > 1)
]
selected_drivers_lap_average_on_hard = selected_drivers_on_hard['LapTime'].mean()

print("The average lap time for all drivers on hard compound: ", lap_average_on_hard % pd.Timedelta(days=1))
print("The average lap time of selected drivers on hard compound: ", selected_drivers_lap_average_on_hard)

#### Visualising Selected Drivers Lap Performances on Hard Compounds (Lap-by-Lap):

In [ ]:
# Adding 'lap time in seconds' column so we can remove the days section of the Timedelta objects from the graph
selected_drivers_on_hard['LapTime_Seconds'] = selected_drivers_on_hard['LapTime'].dt.total_seconds()

laps_hard_fig, laps_hard_ax = plt.subplots()
driver_colours = {
    'BOT' : 'blue',
    'PER' : 'purple',
    'HUL' : 'orange',
    'BOR' : 'red'
}

for driver, group in selected_drivers_on_hard.groupby('Driver'):
    laps_hard_ax.plot(
        range(1, len(group) + 1),
        group['LapTime_Seconds'],
        label=driver,
        color=driver_colours.get(driver)
    )

laps_hard_ax.axhline(y=lap_average_on_hard.total_seconds(), linestyle='--', color='black')
laps_hard_ax.axhline(y=selected_drivers_lap_average_on_hard.total_seconds(), linestyle=':', color='black')

laps_hard_ax.annotate(
    text='Grid-Wide Lap Time Average',
    xy=(1, lap_average_on_hard.total_seconds()),
    xycoords=('axes fraction', 'data'),
    xytext=(5, 0),
    textcoords='offset points',
    va='center',
    clip_on=False
)
laps_hard_ax.annotate(
    text='Selected Drivers Lap Time Average',
    xy=(1, selected_drivers_lap_average_on_hard.total_seconds()),
    xycoords=('axes fraction', 'data'),
    xytext=(5, 0),
    textcoords='offset points',
    va='center',
    clip_on=False
)

plt.title('Selected Drivers\' Lap Times on Hard Compounds', weight='bold', pad=12)
plt.xlabel('Cummulative Laps on Hard Compound')
plt.ylabel('Lap Time (Seconds)')
plt.legend()
plt.show()